# Gold: supplier scorecard
On-time, in-full and OTIF from received POs joined to shipments; lost sales attributed with an **SCD2 as-of join**
(the supplier responsible for the item *on that day*). `reliability_tier` is never used here: it is ground truth (see notebook 90).

In [0]:
%run ./_common

In [0]:
NB = "31_gold_supplier_scorecard"
warmup_end = spark.sql(f"SELECT date_add(MIN(snapshot_date), 28) FROM {CAT}.gold.fact_item_day").first()[0]
spark.sql(f"""
CREATE OR REPLACE TABLE {CAT}.gold.supplier_scorecard AS
WITH deliveries AS (
  SELECT p.supplier_id, p.qty_ordered, s.qty_shipped,
         datediff(s.arrival_date, p.expected_date) AS delay_days
  FROM {CAT}.silver.purchase_orders p
  JOIN {CAT}.silver.shipments s USING (po_id)
  WHERE p.status = 'received'),
perf AS (
  SELECT supplier_id,
         COUNT(*)                                                              AS pos_delivered,
         ROUND(AVG(CASE WHEN delay_days <= 0 THEN 1 ELSE 0 END), 3)            AS on_time_rate,
         ROUND(AVG(CASE WHEN qty_shipped >= qty_ordered THEN 1 ELSE 0 END), 3) AS in_full_rate,
         ROUND(AVG(CASE WHEN delay_days <= 0 AND qty_shipped >= qty_ordered THEN 1 ELSE 0 END), 3) AS otif_rate,
         ROUND(AVG(GREATEST(delay_days, 0)), 2)                                AS avg_delay_days,
         ROUND(SUM(qty_shipped) / SUM(qty_ordered), 3)                         AS unit_fill_rate
  FROM deliveries GROUP BY supplier_id),
lost AS (
  SELECT ps.supplier_id, SUM(f.units_sold) AS units_sold,
         SUM(f.est_lost_units) AS est_lost_units, SUM(f.est_lost_revenue) AS est_lost_revenue
  FROM {CAT}.gold.fact_item_day f
  JOIN {CAT}.silver.products_scd2 ps
    ON ps.item_id = f.item_id AND f.snapshot_date >= ps.valid_from
   AND (ps.valid_to IS NULL OR f.snapshot_date < ps.valid_to)
  WHERE f.snapshot_date >= DATE'{warmup_end}'
  GROUP BY ps.supplier_id)
SELECT p.*, ROUND(l.est_lost_revenue, 0) AS est_lost_revenue,
       ROUND(l.est_lost_units / (l.units_sold + l.est_lost_units), 4) AS lost_sales_rate
FROM perf p LEFT JOIN lost l USING (supplier_id)""")
display(spark.table(f"{CAT}.gold.supplier_scorecard").orderBy("otif_rate"))

In [0]:
n_sc, n_sup = table_count(f"{CAT}.gold.supplier_scorecard"), table_count(f"{CAT}.silver.suppliers")
dq_check("scorecard_covers_all_suppliers", n_sc, n_sc == n_sup, f"{n_sup}")
bad = spark.table(f"{CAT}.gold.supplier_scorecard").where("otif_rate < 0 OR otif_rate > 1 OR otif_rate IS NULL").count()
dq_check("otif_in_range", bad, bad == 0, "0")
dq_finish(NB)